# Phase 2: Data pipeline walkthrough

One section per Phase 2 step, from the synthetic dataset to the rules in pgvector. Each section says what the step does, why it exists and what to look for in the output.

All logic lives in `src/atlas/`; this notebook only calls it and displays results. Run it cell by cell, or top to bottom. With `ATLAS_MOCK=1` and no database it runs offline (CI does this); with a real `.env` it runs against your embedding deployment and database.

## 1. Setup

**What:** load the `atlas` package and show where this run points: mock or live, which embedding model, which database.

**Why:** the same code runs offline in CI and live on your machine. Knowing which one you are looking at comes first. In Phase 2, ingestion once reported success while writing to a different Postgres than expected (case study CS-003).

**Look for:** `mode`, the embed model, and the database host, port and name. The password is never printed: `describe_target()` reads the URL's parts and drops it.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from atlas import store  # noqa: E402  (loads .env)
from atlas.chunking import chunk_rules, content_hash  # noqa: E402
from atlas.corpus import load_rules, sections  # noqa: E402
from atlas.llm import cosine, embed, embed_model_name  # noqa: E402
from atlas.pii import find_pii, load_allowlist  # noqa: E402
from atlas.synth import generate_disputes  # noqa: E402

mode = "mock" if os.environ.get("ATLAS_MOCK") == "1" else "live"
print(f"mode:        {mode}")
print(f"embed model: {embed_model_name()}")
print(f"database:    {store.describe_target()}")

## 2. Synthetic data (spec 2.1)

**What:** `generate_disputes(200, 42)` fills 48 claim templates with fictional merchants, amounts and dates, and labels each case with its reason code.

**Why:** 12 hand-written cases cannot show whether a prompt works. A generator gives a larger labelled set that anyone can rebuild exactly, plus a held-out split that prompt authors never tune on.

**Look for:** 25 cases per code; about 25% held out and about 15% ambiguous; regeneration is identical; no template id appears in both splits; an ambiguous case carries `alt_reason_code`, the expected near miss.

In [ ]:
cases = generate_disputes(200, 42)
df = pd.DataFrame(cases)
pd.crosstab(
    df["label_reason_code"], [df["split"], df["difficulty"]], margins=True, margins_name="total"
)

In [ ]:
print("regenerating with seed 42 gives identical output:", generate_disputes(200, 42) == cases)
print("a different seed gives different output:       ", generate_disputes(200, 7) != cases)

In [ ]:
templates = df.groupby("split")["template_id"].unique().map(sorted)
for split, ids in templates.items():
    print(f"{split:<9} {len(ids)} templates: {', '.join(ids)}")
shared = sorted(set(templates["dev"]) & set(templates["held_out"]))
print("shared between splits:", shared or "none")

In [ ]:
for difficulty in ["standard", "ambiguous"]:
    example = next(c for c in cases if c["difficulty"] == difficulty)  # raw dict: null stays null
    print(f"--- {difficulty}")
    print(json.dumps(example, indent=2))

## 3. PII scan (spec 2.3)

**What:** `find_pii(text)` finds card numbers (Luhn-checked), emails, US phone numbers and SSNs, and masks what it finds. `scripts/scan_data.py` runs it over `data/` and `docs/`.

**Why:** a real card number in the repo is a compliance incident, so the check is automatic and blocks CI.

**Look for:** the test card number is found and shown as first 6 and last 4 digits only; the amount and reason code are not flagged; the email is reported only as `[email]`; the repo scan is clean. The test card is a published network test number read from `config/pii_allowlist.txt`, so no card-like literal sits in this notebook. Only the label and the masked findings are printed, never the raw sample.

In [ ]:
allow = load_allowlist(ROOT / "config/pii_allowlist.txt")
test_card = sorted(v for v in allow if v.isdigit() and len(v) >= 13)[0]

samples = {
    "test card number": f"I was charged twice on card {test_card}.",
    "amount and date": "I paid 1980.13 on 14 May and again on 15 May.",
    "reason code and MCC": "Filed under 13.1 with MCC 5732.",
    "email": "Please reply to cardholder@example.com.",
}
pd.DataFrame(
    [
        {"sample": label, "findings": [(f["type"], f["masked"]) for f in find_pii(text)] or "none"}
        for label, text in samples.items()
    ]
)

In [ ]:
scan = subprocess.run(
    [sys.executable, str(ROOT / "scripts/scan_data.py")], capture_output=True, text=True
)
print(scan.stdout.strip().splitlines()[-1], f"(exit code {scan.returncode})")

## 4. Rules corpus (spec 2.2)

**What:** one markdown file per reason code plus `choosing-between-codes.md`, each with YAML front matter.

**Why:** Phase 3 retrieves these rules and cites them. That makes them a governed asset: each file has an owner, a version for citations (`code@version`) and a review date.

**Look for:** 9 files, each with a version, owner and review date. The "When it does not apply" section names the neighbouring codes, which is where ambiguous cases are decided.

In [ ]:
docs = load_rules(ROOT / "data/rules")
pd.DataFrame(docs)[["file", "code", "version", "owner", "review_by"]]

In [ ]:
rule = next(d for d in docs if d["code"] == "13.1")
print(f"{rule['code']} {rule['title']} (v{rule['version']}): When it does not apply\n")
print(sections(rule["body"])["When it does not apply"])

## 5. Chunking (spec 2.4)

**What:** `chunk_rules(docs)` makes one chunk per `##` section, with its code, title, section, version and a sha256 `content_hash`.

**Why:** retrieval returns a section, not a whole file, so a citation can point to the exact rule. The hash makes ingestion idempotent: a chunk already stored is skipped and never embedded again.

**Look for:** 38 chunks (8 codes x 4 sections, plus 6 pairs). The text starts with the title and heading, so it reads on its own. Changing a single word gives a completely different hash.

In [ ]:
chunks = chunk_rules(docs)
print(len(chunks), "chunks")
pd.DataFrame(chunks).groupby("code").size().rename("chunks").to_frame().T

In [ ]:
chunk = chunks[0]
print(json.dumps({k: v for k, v in chunk.items() if k != "text"}, indent=2))
print()
print(chunk["text"])

In [ ]:
edited = chunk["text"].replace("cardholder", "customer", 1)
print("changed 'cardholder' to 'customer':", edited != chunk["text"])
print("original hash:", chunk["content_hash"])
print("edited hash:  ", content_hash(chunk["code"], chunk["version"], chunk["section"], edited))

## 6. Embeddings (spec 2.4)

**What:** `embed(texts)` turns each text into a vector of 1536 numbers through LiteLLM, the same single model path as `call_model()`.

**Why:** texts with similar meaning get vectors that point in similar directions. That is what lets Phase 3 find the right rule for a claim that uses different words.

**Look for:** vector length 1536. In live mode, the two refund texts are the most similar pair, and the fraud text is further from both. **In mock mode the vectors come from a hash of the text, so the similarities are meaningless:** expect values near 0 for every pair.

In [ ]:
texts = {
    "refund 1": "The merchant confirmed my refund by email, but it never reached my card.",
    "refund 2": "I returned the item and was promised a credit that has not appeared.",
    "fraud": "I do not recognise this online charge and never made the purchase.",
}
vectors = dict(zip(texts, embed(list(texts.values())), strict=True))
print("vector length:", {name: len(v) for name, v in vectors.items()})
similarity = {a: {b: cosine(va, vb) for b, vb in vectors.items()} for a, va in vectors.items()}
pd.DataFrame(similarity).round(3)

In [ ]:
pairs = {(a, b): cosine(vectors[a], vectors[b]) for a in texts for b in texts if a < b}
best = max(pairs, key=pairs.get)
print(f"most similar pair: {best[0]} and {best[1]} ({pairs[best]:.3f})")
if mode == "mock":
    print("mock mode: hash-based vectors, so this ranking means nothing")

## 7. Database (spec 2.4)

**What:** the rules as stored by `scripts/ingest_rules.py` in Postgres with pgvector, plus a preview of the search Phase 3 will build on.

**Why:** this checks the stored result, not just the script's own report: the row count, that every row records its embedding model, and that a plain-language claim finds sensible rules.

**Look for:** 38 rows, all with one `embed_model`. In live mode, the nearest chunks to the refund claim come from 13.1 and 13.6, the pair this claim sits between. By our rules a refund that was only promised, not confirmed in writing, is 13.1, so 13.1 at the top is the right answer. This is a preview by eye, not a measured result; Phase 3 measures retrieval on the dataset. The section is skipped, with the reason, when the database is unreachable (as in CI). The search is also skipped when the stored vectors come from a different model than this run's: vectors from different models cannot be compared.

In [ ]:
reason = store.unavailable_reason()
if reason:
    print("skipped:", reason)
else:
    with store.connect() as conn:
        print("connected:", store.server_info(conn))
        print("rows:     ", store.count_chunks(conn))
        print("models:   ", store.model_counts(conn))

In [ ]:
query = "the merchant promised a refund that never arrived"
if reason:
    print("skipped:", reason)
else:
    with store.connect() as conn:
        stored = store.model_counts(conn)
        if embed_model_name() not in stored:
            print(
                f"skipped search: stored vectors are from {sorted(stored)}, "
                f"this run embeds with {embed_model_name()}"
            )
        else:
            display(pd.DataFrame(store.nearest(conn, embed([query])[0], k=3)).round(3))

## 8. Summary

**What Phase 2 produced**

- **Dataset:** 200 reproducible cases (150 dev, 50 held out, 15% ambiguous) from 48 templates. Held-out templates never appear in dev. `scripts/validate_data.py` checks it in CI.
- **Privacy gate:** `find_pii` and `scripts/scan_data.py` block CI on any card number, email, phone number or SSN in `data/` or `docs/`.
- **Rules corpus:** 9 versioned, owned files that are illustrative, not official network rules.
- **Rules store:** 38 section chunks with content hashes, embedded through `embed()` and stored in pgvector with their model name. Running ingestion again embeds nothing.
- **Documentation:** `docs/data-card.md` (what the data is and is not for) and one spec per step, each with its decisions.

**What Phase 3 will use**

- `rule_chunks` and the cosine search previewed in section 7, to retrieve rules for a claim and cite them as `code@version`.
- The dev split to tune retrieval and prompts, and the held-out split to check that the results carry over.
- Open items from `docs/roadmap.md`: filter to the current rule version, ingest the intro of `choosing-between-codes.md`, and add a vector index if the corpus grows.